In [1]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
import ast
import torch
from tqdm import tqdm
tqdm.pandas(desc="Processing Rows")
import os
import time
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
hf_token = "os.environ['HF_TOKEN']" 
import gc
gc.collect()
import torch
from collections import Counter
from sklearn.metrics import confusion_matrix

/apps/common/software/Miniforge3/25.11.0-1-jupyter-base/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# !ls /common/data/models/

In [3]:
df=pd.read_csv("sample_IPAWS_data.csv")

In [78]:
df.columns

Index(['identifier', 'sent', 'status', 'msgType', 'senderName', 'event',
       'responseType', 'urgency', 'severity', 'certainty', 'headline',
       'description', 'instruction', 'areaDesc', 'date', 'time'],
      dtype='object')

In [6]:
natural_sample=pd.read_csv("natural_sample.csv")
balanced_sample=pd.read_csv("balanced_sample.csv")

In [10]:
label_map = {
    "A": "Extreme",
    "B": "Severe",
    "C": "Moderate",
    "D": "Minor",

}
choices = list(label_map)

template = """Classify the emergency message severity.

A: extraordinary threat to life or property
B: significant threat to life or property
C: possible threat to life or property
D: minimal/no known threat to life or property


Message:
{}

Answer only A, B, C, D, or E."""
severity_num = {
    "Minor": 1,
    "Moderate": 2,
    "Severe": 3,
    "Extreme": 4
}

def initalSetup(name):
    
    model_name = f"/common/data/models/{name}"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    
    model.eval()
    
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    
    
    model.eval()
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    choice_ids = {
        c: tokenizer(" " + c, add_special_tokens=False,
                     return_tensors="pt").input_ids[0]
        for c in choices
    }
    return name, model, tokenizer, choice_ids
    
def classify(msg, model, tokenizer, choice_ids):
    messages = [{"role": "user", "content": template.format(msg)}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    pids = tokenizer(
        prompt,
        add_special_tokens=False,
        truncation=True,
        return_tensors="pt"
    ).input_ids[0]

    seqs = [torch.cat([pids, choice_ids[c]]) for c in choices]

    batch = torch.nn.utils.rnn.pad_sequence(
        seqs, batch_first=True,
        padding_value=tokenizer.pad_token_id
    ).to(model.device)

    lengths = torch.tensor([len(x) for x in seqs], device=model.device)
    mask = (
        torch.arange(batch.shape[1], device=model.device)[None, :]
        < lengths[:, None]
    ).long()

    with torch.inference_mode():
        logits = model(input_ids=batch, attention_mask=mask).logits

    p = len(pids)
    scores = {}

    for i, c in enumerate(choices):
        ids = choice_ids[c].to(model.device)
        lp = torch.log_softmax(
            logits[i, p-1:p+len(ids)-1], dim=-1
        )
        scores[c] = lp.gather(
            1, ids.unsqueeze(-1)
        ).sum().item()

    return label_map[max(scores, key=scores.get)]


def evaluate_severity(sample, fn, model, tokenizer,choice_ids):
    

    sample["prediction"] = [
        classify(x, model, tokenizer, choice_ids) for x in sample["description"]
    ]

    sample["true_num"] = sample["severity"].map(severity_num)
    sample["pred_num"] = sample["prediction"].map(severity_num)
    sample["error"] = sample["pred_num"] - sample["true_num"]

    valid = sample.dropna(subset=["true_num", "pred_num"])
    sample.to_csv(f"{fn}.csv", index=False)
    results = pd.DataFrame.from_dict({
        "n": len(sample),
    
        "mean_signed_error": valid["error"].mean(),
        "over": (valid["error"] > 0).mean(),
        "correct": (valid["error"] == 0).mean(),
        "under": (valid["error"] < 0).mean(),
    
        "actual_minor": (sample["severity"] == "Minor").sum(),
        "actual_moderate": (sample["severity"] == "Moderate").sum(),
        "actual_severe": (sample["severity"] == "Severe").sum(),
        "actual_extreme": (sample["severity"] == "Extreme").sum(),
    
        "pred_minor": (sample["prediction"] == "Minor").sum(),
        "pred_moderate": (sample["prediction"] == "Moderate").sum(),
        "pred_severe": (sample["prediction"] == "Severe").sum(),
        "pred_extreme": (sample["prediction"] == "Extreme").sum(),
    }, orient="index", columns=["value"])
    sample.to_csv(f"Results/Experiment1/Part1/{fn}.csv", index=False)
    return sample, results

In [13]:
# small_names=["google--gemma-3-4b-it", "meta-llama--Llama-3.2-3B-Instruct", "microsoft--Phi-4-mini-instruct"]
large_names=["google--gemma-3-12b-it", "meta-llama--Llama-3.1-8B-Instruct", "microsoft--phi-4"]
full=[]
for name in large_names:
    print(name)
    name, model, tokenizer, choice_ids=initalSetup(name)
    nat=[True, False]
    dfs=[]
    for i in nat:
        fn=f"{name}_{"natural" if i else "balanced"}"
        if i:
            sample_df=natural_sample.copy()
        else:
            sample_df=balanced_sample.copy()   
        sample, results=evaluate_severity(sample_df, fn, model, tokenizer, choice_ids)
        results.columns = [fn] 
        dfs.append(results)
    del model, tokenizer, choice_ids
    torch.cuda.empty_cache()
    full.append(pd.concat([dfs[0], dfs[1]], axis=1))

full = pd.concat(full, axis=1)
full

google--gemma-3-12b-it


Loading weights: 100%|██████████| 1065/1065 [00:28<00:00, 37.72it/s] 


meta-llama--Llama-3.1-8B-Instruct


Loading weights: 100%|██████████| 291/291 [00:18<00:00, 15.53it/s]


microsoft--phi-4


Loading weights: 100%|██████████| 243/243 [00:32<00:00,  7.41it/s]


,google--gemma-3-12b-it_natural,google--gemma-3-12b-it_balanced,meta-llama--Llama-3.1-8B-Instruct_natural,meta-llama--Llama-3.1-8B-Instruct_balanced,microsoft--phi-4_natural,microsoft--phi-4_balanced
n,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000
mean_signed_error,-0.099,-0.012,0.383,0.314,0.326,0.368
over,0.172,0.289,0.495,0.394,0.414,0.414
correct,0.556,0.444,0.382,0.419,0.467,0.410
under,0.272,0.267,0.123,0.187,0.119,0.176
actual_minor,72.000,250.000,72.000,250.000,72.000,250.000
actual_moderate,503.000,250.000,503.000,250.000,503.000,250.000
actual_severe,405.000,250.000,405.000,250.000,405.000,250.000
actual_extreme,20.000,250.000,20.000,250.000,20.000,250.000
actual_unknown,0.000,0.000,0.000,0.000,0.000,0.000


In [87]:
# Llama-3.2-3B-Instruct exhibits a strong severity compression bias, disproportionately mapping emergency messages to the “Severe” category, with an upward bias under naturally imbalanced data.

In [88]:


# labels = ["Minor", "Moderate", "Severe", "Extreme"]
# print(confusion_matrix(
#     balanced["severity"],
#     balanced["prediction"],
#     labels=labels
# ))